# EY Low-Vol Strategy — Backtest Demo

This notebook demonstrates the full backtest pipeline for the
**Earnings Yield + Low-Volatility** strategy on A-share stocks.

## Contents
1. Data Loading
2. Strategy Execution
3. Performance Analysis
4. Equity Curve Comparison

In [ ]:
# Install dependencies if needed
# !pip install -r ../requirements.txt

## 1. Data Loading

Load OHLCV data for HS300 constituent stocks and the index.

In [ ]:
import sys
sys.path.insert(0, '..')

from strategy_ey_lowvol import (
    load_data, load_index, load_eps_history,
    ey_lowvol_strategy, ALL_CODES
)

codes = [c for c in ALL_CODES if c != '601088']
common_dates, closes, opens, highs, lows = load_data(codes)
idx_bars = load_index()
idx_close_map = {b[0]: b[1] for b in idx_bars}
eps_hist = load_eps_history(codes)

print(f"Loaded {len(common_dates)} days, {len(eps_hist)} stocks with EPS")

## 2. Strategy Execution

Run with default parameters from README:
- `lookback=160`, `weight_pow=4.5`, `ey_pow=0.5`
- MA timing: fast=50, slow=200, exposure=0.5

In [ ]:
result = ey_lowvol_strategy(
    common_dates, closes, opens, highs, lows, idx_close_map, eps_hist,
    lookback=160, weight_pow=4.5, ey_pow=0.5,
    ma_fast=50, ma_slow=200, ma_exposure=0.5,
)

## 3. Performance Analysis

In [ ]:
print(f"Annual Return: {result['annual']*100:+.2f}%")
print(f"Max Drawdown : {result['max_dd']*100:.2f}%")
print(f"Sharpe Ratio: {result['sharpe']:.3f}")
print(f"Total Trades : {result['n_trades']}")

## 4. Equity Curve Comparison

In [ ]:
import matplotlib.pyplot as plt

bh_eq = [1.0]
for i in range(1, len(common_dates)):
    d, pd = common_dates[i], common_dates[i-1]
    if d in idx_close_map and pd in idx_close_map and idx_close_map[pd] > 0:
        ret = (idx_close_map[d] - idx_close_map[pd]) / idx_close_map[pd]
        bh_eq.append(bh_eq[-1] * (1 + ret))
    else:
        bh_eq.append(bh_eq[-1])

dates = common_dates[:len(result['equity_curve'])]

plt.figure(figsize=(12, 6))
plt.plot(dates, [e/1e6 for e in result['equity_curve']], label='Strategy', linewidth=1.5)
plt.plot(dates, [e for e in bh_eq[:len(dates)]], label='Buy & Hold', linewidth=1.5, alpha=0.7)
plt.title('EY Low-Vol Strategy vs Buy & Hold (HS300)')
plt.xlabel('Date')
plt.ylabel('Normalized Equity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()